# Question
What is the probability that an AI Agent that is randomly moving in PacMan picks up it's tenth orb in 10 seconds?

### Simulation
I will simulate this by giving an AI Agent access to PacMan or some form of it and only tell it that it can move around.

In [131]:
from IPython.display import clear_output, display
import matplotlib.pyplot as plt
import gymnasium as gym
import seaborn as sns
import pandas as pd
import numpy as np
import pathlib
import ale_py
import time
import csv

In [132]:
sns.set_theme('paper')
sns.set_style('darkgrid')
sns.set_palette(["#ff9a2a"])
gym.register_envs(ale_py)

In [133]:
env = gym.make('ALE/Pacman-v5', render_mode='rgb_array')
observation, info = env.reset()

In [134]:
def frameinit():
  fig, ax = plt.subplots(figsize=(6, 8))
  ax.axis('off')

In [135]:
max_steps_10s = 150
target_fps = 30
frame_duration = 1 / target_fps

In [136]:
def run_trials(
  num_trials=100,
  target_orbs=10,
  render=True,
  render_every=50,
  filename="pacman_trials.csv",
):
  successful_steps = []
  failures = 0
  trial_records = []

  for trial in range(num_trials):
    observation, info = env.reset()
    orbs_collected = 0
    total_reward = 0
    should_render = render and (trial % render_every == 0)

    for step in range(max_steps_10s):
      action = env.action_space.sample()
      observation, reward, terminated, truncated, info = env.step(action)
      total_reward += reward

      if reward > 0:
        orbs_collected += 1

      if should_render:
        clear_output(wait=True)
        ax.clear()
        ax.imshow(env.render())
        ax.set_title(
          f"Trial: {trial + 1}/{num_trials} | Step: {step}/{max_steps_10s} | Orbs: {orbs_collected}/{target_orbs}",
          fontsize=12,
        )
        ax.axis("off")
        display(fig)
        time.sleep(frame_duration)

      if orbs_collected == target_orbs:
        successful_steps.append(step)
        trial_records.append([trial + 1, True, step, orbs_collected, total_reward])
        break

      if terminated or truncated:
        break

    if orbs_collected < target_orbs:
      failures += 1
      trial_records.append([trial + 1, False, step, orbs_collected, total_reward])

  with open(filename, mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["trial", "success", "steps_taken", "orbs_collected", "total_reward"])
    writer.writerows(trial_records)

  plt.close(fig)
  env.close()

  clear_output(wait=True)
  probability = (len(successful_steps) / num_trials) * 100
  print(f"Completed {num_trials} trials. Data saved to '{filename}'.")
  print(
    f"Successes ({target_orbs}th orb in 10s): {len(successful_steps)} ({probability:.1f}%)"
  )
  print(f"Failures (timed out or caught by ghost): {failures}")

  return successful_steps

#run_trials(num_trials=5000, target_orbs=10, render=True, render_every=550)

In [137]:
df = pd.read_csv(pathlib.Path('./pacman_trials.csv'))
successful_steps = df[df["success"] == True]["steps_taken"]

mean = successful_steps.mean()
std = successful_steps.std()

In [138]:
def showplt():
  fig, ax = plt.subplots(figsize=(9, 5))

  sns.histplot(
    successful_steps,
    kde=True,
    stat="density",
    bins=20,
    color="#ff9a2a",
    ax=ax,
    label="Actual Simulation Data",
  )

  x = np.linspace(28, 150, 200)
  y = (1 / (std * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mean) / std) ** 2)
  ax.plot(
    x, y, color="#333333", linestyle="--", linewidth=2, label="Normal Distribution Curve"
  )

  ax.axvspan(0, 28, color="#888888", alpha=0.2, label="Static Intro (Steps 0-28)")
  ax.axvline(mean, color="#d9534f", linestyle=":", linewidth=2, label=f"Mean: {mean:.1f} steps")
  ax.axvline(149, color="#2ca02c", linestyle="-.", linewidth=1.5, label="10s Cutoff (Step 149)")

  ax.set_xlim(20, 155)
  ax.set_title("Steps Taken to Collect 10th Orb (n = 756 Successful Trials)", fontsize=14, pad=12)
  ax.set_xlabel("Total Environment Steps", fontsize=12)
  ax.set_ylabel("Density", fontsize=12)
  ax.legend(loc="upper left")

  plt.tight_layout()

  plt.savefig("python_graph.png", dpi=300, bbox_inches="tight")
  plt.show()

# Conclusion
Across 5,000 simulated trials, a randomly moving AI agent has a 15.1% probability (756 out of 5,000 runs) of collecting its 10th orb within 10 seconds (150 steps).   For the successful trials, the agent took an average of 105.9 steps ($\mu = 105.9$) with a standard deviation of 26.6 steps ($\sigma = 26.6$), meaning most successful runs finished between 79.3 and 132.5 steps ($\pm 1\sigma$). However, the data does not follow a pure normal distribution: it is bounded on the left by the 28-step static intro (fastest run at 56 steps) and right-truncated at step 149 by the 10-second time limit, creating a bimodal spike of agents barely reaching the 10th orb right before time expires.<br>
<img src="https://raw.githubusercontent.com/hxproxy/AI-Class/refs/heads/main/Semester1/Unit2/Atari/python_graph.png" width="669" height="368.5" alt="graph">